In [3]:
!pip install -q faiss-cpu sentence-transformers transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train = pd.read_csv('C:\\Users\\SAGAR\\OneDrive\\Desktop\\dl_genai\\dl-genai-project\\data\\train.csv')   # <-- apna path daalna

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))    # KB[i] = row i ka CORRECT answer text

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
print("Knowledge base created. KB size:", len(kb))

Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base created. KB size: 2000


In [7]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']),
              str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])   # ground-truth option ka TEXT

res1 = zs(prompt_150, candidate_labels=labels_150)
# correct option ka score dhundo (labels shuffle hoke aate hain score-order mein)
correct_score = res1['scores'][res1['labels'].index(ans_150)]
print("ANSWER Q1:", round(correct_score, 3))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ANSWER Q1: 0.384


In [8]:
q_emb = model.encode([prompt_150])
D, I = index.search(np.array(q_emb), k=10)
retrieved_indices = I[0].tolist()
print("retrieved KB indices:", retrieved_indices)

if 150 in retrieved_indices:
    rank = retrieved_indices.index(150) + 1    # rank 1-based
    print("ANSWER Q2:", rank)
else:
    print("ANSWER Q2: true doc (KB index 150) top-10 mein NAHI hai")

retrieved KB indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
ANSWER Q2: 10


In [9]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# scores ke hisaab se sort (descending), phir true doc ka naya rank
order = np.argsort(-ce_scores)
sorted_kb_indices = [retrieved_indices[j] for j in order]
rank_ce = sorted_kb_indices.index(150) + 1
print("ANSWER Q3:", rank_ce)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

ANSWER Q3: 1


In [10]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])
q_emb42 = model.encode([prompt_42])
D42, I42 = index.search(np.array(q_emb42), k=5)
docs_5 = [kb[i] for i in I42[0]]
concat = " ".join(docs_5)
rag_str_42 = f"Context: {concat} Question: {prompt_42}"

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = bert_tok(rag_str_42, truncation=False)["input_ids"]
print("ANSWER Q4:", len(tokens))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ANSWER Q4: 216


In [11]:
true_doc = kb[150]
rag_str = f"Context: {true_doc} Question: {prompt_150}"
res5 = zs(rag_str, candidate_labels=labels_150)
score5 = res5['scores'][res5['labels'].index(ans_150)]
print("ANSWER Q5:", round(score5, 3))

ANSWER Q5: 0.989


In [12]:
bad_doc = kb[999]
adv_str = f"Context: {bad_doc} Question: {prompt_150}"
res6 = zs(adv_str, candidate_labels=labels_150)
score6 = res6['scores'][res6['labels'].index(ans_150)]
print("ANSWER Q6:", round(score6, 3))

ANSWER Q6: 0.529


In [13]:
hits = 0
prompts_100 = [str(train.iloc[i]['prompt']) for i in range(100)]
embs = model.encode(prompts_100, show_progress_bar=False)
D, I = index.search(np.array(embs), k=5)

for i in range(100):
    correct_text = str(train.iloc[i][train.iloc[i]['answer']])
    retrieved_docs = [kb[j] for j in I[i]]
    if any(correct_text in doc for doc in retrieved_docs):   # "found inside" = substring
        hits += 1

print("ANSWER Q7:", round(100.0 * hits / 100, 1))

ANSWER Q7: 73.0


In [14]:
OPTS = ['A','B','C','D','E']
total = 0.0

for i in range(20):
    row = train.iloc[i]
    prompt = str(row['prompt'])

    # 1) Retrieve top-5
    q = model.encode([prompt])
    _, Ii = index.search(np.array(q), k=5)
    docs5 = [kb[j] for j in Ii[0]]

    # 2) Rerank -> best doc
    pairs = [[prompt, d] for d in docs5]
    scores = cross_encoder.predict(pairs)
    best_doc = docs5[int(np.argmax(scores))]

    # 3) Augment
    rag = f"Context: {best_doc} Question: {prompt}"

    # 4) Predict (zero-shot, 5 options)
    labels = [str(row[o]) for o in OPTS]
    res = zs(rag, candidate_labels=labels)

    # 5) top-3 letters -> MAP@3 for this row
    #    res['labels'] score-order mein hain; unhe letters mein map karo
    top3_letters = [OPTS[labels.index(l)] for l in res['labels'][:3]]
    gt = row['answer']
    for r, p in enumerate(top3_letters):
        if p == gt:
            total += 1.0 / (r + 1)
            break

print("ANSWER Q8:", round(total / 20, 3))

ANSWER Q8: 0.975
